# 4.13 · 梯度提升 / Gradient Boosting + XGBoost ⭐

> **课程定位 / Where this fits**
> **Part 4 第 13 课, 树家族的巅峰**。RF(4.12) 并行平均**降方差**; GBDT 串行纠错**降偏差**: 每棵新树拟合前面模型的**残差(负梯度)**, 逐步逼近。**XGBoost/LightGBM 是 Kaggle 表格赛常年冠军**。本课讲清 boosting 数学 + 从零实现 + XGBoost 实战。
> RF averages to cut variance; GBDT corrects sequentially to cut bias — each new tree fits the residual (negative gradient). XGBoost/LightGBM dominate Kaggle tabular.

> 💡 **面试相关 / Interview-relevant**
> - "梯度提升的核心思想" ★★★★★（拟合残差/负梯度）
> - "GBDT 和随机森林区别" ★★★★★（boosting vs bagging）
> - "学习率的作用" ★★★★
> - "XGBoost 比 sklearn GBDT 强在哪" ★★★★（二阶 + 正则 + 工程）
> - "GBDT 怎么防过拟合" ★★★★（lr + early stopping + 正则）

---

## 学习目标 / Learning Objectives
1. 理解 **boosting**：加法模型, 每棵树拟合当前残差。
2. **梯度视角**：拟合残差 = 拟合损失的负梯度（接 0.8/0.10）。
3. 从零实现一个梯度提升回归器。
4. 学习率 × 树数的**权衡**, early stopping 防过拟合。
5. **XGBoost** 实战 + 它的三大改进（二阶 Taylor + 正则 + 工程）。

## 目录 / TOC
1. [boosting: 加法纠错 ⭐](#1)
2. [梯度视角: 拟合负梯度 ⭐](#2)
3. [💎 从零实现 GBDT](#3)
4. [对照 sklearn + 学习率](#4)
5. [early stopping 防过拟合 ⭐](#5)
6. [XGBoost 实战 ⭐](#6)
7. [树家族大对决](#7)
8. [小结 + Part 4 树族总结](#8)


<a id="1"></a>
## 1. boosting: 加法纠错 ⭐ / Boosting = Additive Correction

**核心思想**：不像 RF 那样并行造很多独立树, boosting **串行**地造树, **每棵新树专门修正前面所有树的错误**：

$$F_0(\mathbf{x}) = \bar{y}, \qquad F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta\, h_m(\mathbf{x})$$

其中 $h_m$ 是第 $m$ 棵树, 它**拟合当前残差** $r_i = y_i - F_{m-1}(\mathbf{x}_i)$, $\eta$ 是学习率。

**直觉**:
- 第 1 棵树预测得粗糙 → 留下残差
- 第 2 棵树**专门学这些残差**（前面错在哪）
- 第 3 棵树学第 2 棵之后剩下的残差……
- 累加起来逐步逼近真值

| | 随机森林 (bagging) | GBDT (boosting) |
|---|---|---|
| 顺序 | 并行独立 | **串行依赖** |
| 每棵树学 | 原始 y (各看 bootstrap) | **当前残差** |
| 树的深度 | 深(强学习器) | **浅(弱学习器, 如 depth=3)** |
| 降什么 | 方差 | **偏差** |

**弱学习器变强**: 每棵浅树很弱（高偏差）, 但**几百棵依次纠错累加 → 强模型**（偏差骤降）。
Each shallow tree is weak, but hundreds correcting each other in sequence make a strong model.


<a id="2"></a>
## 2. 梯度视角: 拟合负梯度 ⭐ / The Gradient View

为什么叫"**梯度**提升"？因为"拟合残差"其实是"**拟合损失函数的负梯度**"的特例。

对平方损失 $L = \frac{1}{2}(y - F)^2$：
$$-\frac{\partial L}{\partial F} = y - F = \text{残差}$$

**所以拟合残差 = 沿损失的负梯度方向走一步**（0.8/0.10 梯度下降, 但这里是在"函数空间"下降）。这个视角的威力: **换损失函数**就能 boosting 任何东西——
- 平方损失 → 拟合残差（回归）
- 绝对损失 → 拟合残差的符号（稳健回归）
- 对数损失 → 分类（Part 5.7/5.8）
- pinball 损失 → 分位数（4.14）

GBDT 是一个**通用框架**: 任意可导损失, 用树拟合它的负梯度。这就是 Friedman 1999 的"梯度提升机"。
Fitting residuals = following the loss's negative gradient in function space. Swap the loss, boost anything.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

df = sns.load_dataset("diamonds").sample(8000, random_state=0).reset_index(drop=True)
X = df[["carat","depth","table","x","y","z"]].values
y = df["price"].values
feat_names = ["carat","depth","table","x","y","z"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
print(f"Diamonds: {X.shape}")


<a id="3"></a>
## 3. 从零实现 GBDT / From Scratch


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# 从零梯度提升 (平方损失 → 拟合残差) / from-scratch GBDT
class MyGBDT:
    def __init__(self, n_estimators=100, lr=0.1, max_depth=3):
        self.n_estimators, self.lr, self.max_depth = n_estimators, lr, max_depth
    def fit(self, X, y):
        self.F0 = y.mean()                       # 初始预测 = 均值
        F = np.full(len(y), self.F0)
        self.trees = []
        for _ in range(self.n_estimators):
            residual = y - F                     # 负梯度(平方损失) = 残差
            tree = DecisionTreeRegressor(max_depth=self.max_depth).fit(X, residual)
            F += self.lr * tree.predict(X)        # 加上 lr × 新树的残差拟合
            self.trees.append(tree)
        return self
    def predict(self, X):
        return self.F0 + self.lr * sum(t.predict(X) for t in self.trees)

my = MyGBDT(n_estimators=100, lr=0.1, max_depth=3).fit(X_tr, y_tr)
from sklearn.metrics import r2_score
print(f"从零 GBDT test R² = {r2_score(y_te, my.predict(X_te)):.4f}")

from sklearn.ensemble import GradientBoostingRegressor
sk = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0).fit(X_tr, y_tr)
print(f"sklearn GBDT  test R² = {sk.score(X_te, y_te):.4f}")
print("→ 从零实现和 sklearn 接近; 核心就是'每棵树拟合当前残差'这一行")


In [ ]:
# 可视化 boosting 逐步纠错 (1D) / visualize boosting correcting step by step
xc = df["carat"].values.reshape(-1,1); yc = y
order = np.argsort(xc.ravel()); xc, yc = xc[order], yc[order]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, [1, 10, 100]):
    g = GradientBoostingRegressor(n_estimators=n, learning_rate=0.3, max_depth=2, random_state=0).fit(xc, yc)
    ax.scatter(xc, yc, alpha=0.08, s=6)
    ax.plot(xc, g.predict(xc), "r-", lw=2)
    ax.set_title(f"{n} 棵树后")
plt.tight_layout(); plt.show()
print("1 棵树: 粗糙阶梯; 10 棵: 渐细; 100 棵: 平滑贴合 — 逐步纠错的累积效果")


<a id="4"></a>
## 4. 学习率 × 树数 / Learning Rate × Trees

**核心权衡**（boosting 的灵魂超参对）：
- **学习率 $\eta$ 小** → 每棵树贡献小 → 需要**更多树** → 但**泛化更好**（小步慢走不易过拟合）
- **学习率大** → 收敛快 → 但易过拟合 / 跳过最优

**黄金法则**: **小学习率 (0.01-0.1) + 多树 + early stopping**。lr 和 n_estimators 是**此消彼长**的——降 lr 必须升 n_estimators。
The golden rule: small learning rate + many trees + early stopping. lr and n_estimators trade off.


In [ ]:
# lr 影响 / learning rate effect
for lr in [0.01, 0.1, 0.5, 1.0]:
    g = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, max_depth=3, random_state=0)
    train_r2 = g.fit(X_tr, y_tr).score(X_tr, y_tr)
    test_r2 = g.score(X_te, y_te)
    print(f"lr={lr:<5} train R²={train_r2:.4f}  test R²={test_r2:.4f}  "
          f"{'(大lr: train高test低=过拟合迹象)' if lr>=0.5 else ''}")
print("\n小 lr 泛化好但需更多树; 大 lr 快但易过拟合(train-test 缺口大)")


<a id="5"></a>
## 5. early stopping 防过拟合 ⭐ / Early Stopping

**RF 树多不过拟合, 但 GBDT 树多会过拟合**（4.12 的关键区别）——因为每棵树都在更紧地拟合训练集, 最终开始追噪声。

**Early stopping**: 监控验证集分数, **验证分数不再提升就停止加树**。这是 boosting 防过拟合的核心机制。


In [ ]:
# 用 staged_predict 看每棵树后的 train/test 误差 / track error per boosting stage
from sklearn.metrics import mean_squared_error

g = GradientBoostingRegressor(n_estimators=500, learning_rate=0.1, max_depth=4, random_state=0)
g.fit(X_tr, y_tr)

train_err = [mean_squared_error(y_tr, p) for p in g.staged_predict(X_tr)]
test_err = [mean_squared_error(y_te, p) for p in g.staged_predict(X_te)]
best_iter = np.argmin(test_err) + 1

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_err, label="train MSE")
ax.plot(test_err, label="test MSE")
ax.axvline(best_iter, color="r", ls="--", label=f"最优树数={best_iter} (early stop 点)")
ax.set_xlabel("树数 (boosting 轮)"); ax.set_ylabel("MSE"); ax.legend()
ax.set_title("GBDT: train 一直降, 但 test 先降后升(过拟合) → early stopping")
plt.tight_layout(); plt.show()
print(f"train MSE 单调下降; test MSE 在 ~{best_iter} 棵树后开始回升 = 过拟合")
print("→ early stopping 在 test 拐点停止; sklearn 用 n_iter_no_change, XGBoost 用 early_stopping_rounds")


<a id="6"></a>
## 6. XGBoost 实战 ⭐ / XGBoost in Action

**XGBoost (eXtreme Gradient Boosting)** 是 GBDT 的工程巅峰, 三大改进：

| 改进 | sklearn GBDT | XGBoost |
|---|---|---|
| **二阶 Taylor** | 只用一阶梯度 | **用二阶(梯度+Hessian)** → 更准的步长(0.10 牛顿法思想) |
| **正则化** | 弱 | **显式 L1/L2 + 叶子数惩罚** → 抗过拟合 |
| **工程** | 单线程慢 | **并行/缓存/稀疏感知/GPU** → 快几个数量级 |

XGBoost 目标函数（二阶近似 + 正则, 接 0.8 Taylor + 0.10 牛顿 + 4.4 正则）：
$$\text{Obj} \approx \sum_i \big[g_i f(\mathbf{x}_i) + \tfrac{1}{2}h_i f(\mathbf{x}_i)^2\big] + \gamma T + \tfrac{1}{2}\lambda\|w\|^2$$
（$g_i, h_i$ = 一阶/二阶梯度, $T$ = 叶子数, $\lambda$ = L2）


In [ ]:
import xgboost as xgb

# XGBoost 回归 + early stopping / XGBoost regression
dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=feat_names)
dtest = xgb.DMatrix(X_te, label=y_te, feature_names=feat_names)

xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.1, max_depth=4,
    reg_lambda=1.0, reg_alpha=0.0,        # L2 / L1 正则
    early_stopping_rounds=20, random_state=0, n_jobs=-1,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
print(f"XGBoost test R² = {xgb_model.score(X_te, y_te):.4f}")
print(f"early stopping 在第 {xgb_model.best_iteration} 棵树停止 (而非全部 500)")

# 对比 sklearn GBDT / vs sklearn GBDT
print(f"sklearn GBDT test R² = {GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=0).fit(X_tr, y_tr).score(X_te, y_te):.4f}")


In [ ]:
# XGBoost 特征重要性 / feature importance
imp = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
imp.plot(kind="barh", ax=ax); ax.invert_yaxis()
ax.set_title("XGBoost 特征重要性")
plt.tight_layout(); plt.show()
print("💡 LightGBM(leaf-wise 生长) 和 CatBoost(原生类别+ordered boosting) 是 XGBoost 的两个强力同类")
print("   三者将在 Part 5.8-5.11 分类场景详细对比")


<a id="7"></a>
## 7. 树家族大对决 / Tree Family Showdown


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import time

contestants = {
    "单棵树":        DecisionTreeRegressor(max_depth=10, random_state=0),
    "随机森林":      RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "sklearn GBDT":  GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=0),
    "XGBoost":       xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=0, n_jobs=-1),
}
print(f"{'模型':<16} {'test R²':>9} {'训练秒':>8}")
for name, m in contestants.items():
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr)
    dt = time.perf_counter() - t0
    print(f"{name:<18} {m.score(X_te, y_te):>9.4f} {dt:>8.2f}")
print("\n精度: 单树 < 森林 < GBDT ≈ XGBoost; XGBoost 通常精度最高且训练快")
print("这就是 Kaggle 表格赛 XGBoost/LightGBM 常年霸榜的原因")


<a id="8"></a>
## 8. 小结 + Part 4 树族总结 / Summary

```
GBDT = boosting: 串行加法, 每棵新树拟合当前残差(=负梯度)
梯度视角 ⭐: 拟合残差 = 沿损失负梯度走(函数空间梯度下降); 换损失 boost 任何东西
浅树(弱学习器) 依次纠错 → 强模型; 降偏差(vs RF 降方差)
学习率 × 树数: 小 lr + 多树 + early stopping (黄金法则)
GBDT 树多会过拟合(vs RF) → early stopping 必需
XGBoost 三改进: 二阶 Taylor(牛顿) + 显式正则 + 工程优化
树族精度: 单树 < 森林 < GBDT ≈ XGBoost
```

### Part 4 树族脉络
```
决策树(4.11): 弱, 高方差 → 两条集成路:
  ├── bagging → 随机森林(4.12): 并行平均降方差, 零调参安全
  └── boosting → GBDT/XGBoost(4.13): 串行纠错降偏差, 精度之王
```

### 💡 面试速查
1. **GBDT 核心**: 每棵新树拟合前面模型的残差(=负梯度)
2. **GBDT vs RF**: 串行降偏差 vs 并行降方差; 浅树 vs 深树
3. **小 lr + 多树 + early stopping**: boosting 黄金法则
4. **GBDT 树多会过拟合** (RF 不会) → early stopping
5. **XGBoost 三改进**: 二阶 Taylor + 正则 + 工程并行

### 下一节
**4.14 分位数回归**——前面都预测均值(条件期望)。但要"95% 情况下房价不超过多少"(预测区间)? 改损失为 pinball loss, 直接预测分位数。
